# [LES - nut- WALE] PitzDaily

## Preamble

In [1]:
# Standard Library
import sys
import os
from pathlib import Path
import foamnordic as fno
import numpy as np
import matplotlib.pyplot as plt
import onsaemiro as osm

### Directory & Path

In [2]:
# FoamNordic Project Directory
PROJECT_DIR = Path("/scratch/<allocation-account>/<user>")
CASE_TYPE = "les"
CASE_NAME = "pitzDailyWALE"
BASE_DIR = PROJECT_DIR / "Codes" / "FoamNordic"
MAIN_DIR = BASE_DIR / "foamnordic_tutorials" / "incompressible"
OF_SCRIPT_DIR = BASE_DIR / "openfoam_tutorials" / CASE_TYPE / CASE_NAME

# Output Directory
MODEL_DIR = MAIN_DIR / "model"
OUTPUT_DIR = MAIN_DIR / "output"

for directory in [MODEL_DIR, OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

### Configuration

In [3]:
# DataGraph Configuration
FIG_X, FIG_Y = 3.5, 2.55
FIGURE_SIZE = (FIG_X, FIG_Y)
PALETTE = osm.get_palette("OKABE_ITO")
osm.set_style(figure_size=FIGURE_SIZE)

In [ ]:
# HPC configuration
ACCOUNT = "<allocation-account>"
PARTITION = "small"
TIME = "00:15:00"
N_NODES = 1
N_TASKS = 1
CPUS_PER_TASK = 1
MEM_PER_CPU = "2G"

# FoamNordic Slurm configuration
of_scheduler = fno.Slurm.openfoam(
    nodes=N_NODES,
    ntasks=N_TASKS,
    cpus_per_task=CPUS_PER_TASK,
    mem_per_cpu=MEM_PER_CPU,
)

model_scheduler = fno.Slurm.model(
    cpus_per_task=1,
    mem_per_cpu=MEM_PER_CPU
)

scheduler = fno.Slurm(
    account=ACCOUNT,
    partition=PARTITION,
    time=TIME,
    openfoam=of_scheduler,
    model=model_scheduler
)

In [5]:
# FoamNordic configuration
SEED = 42
key = fno.Random.key(seed=SEED, scope="global")

## Example - Closure Modelling

### Smagorinsky Closure

In [6]:
# WALE model coefficient
C_W = 0.325

# WALE function for subgrid-scale turbulence modelling
def wale_function(velocity_grad, delta, C_w=C_W):
    gradient_squared = fno.Math.matmul(velocity_grad, velocity_grad)
    traceless_squared = fno.Math.dev(fno.Math.symm(gradient_squared))
    strain = fno.Math.symm(velocity_grad)

    sd2 = fno.Math.maximum(fno.Math.ddot(traceless_squared, traceless_squared), 0.0)
    s2 = fno.Math.maximum(fno.Math.ddot(strain, strain), 0.0)

    numerator = sd2**1.5
    denominator = s2**2.5 + sd2**1.25
    safe_denominator = fno.Math.where(denominator > 0.0, denominator, 1.0)

    nut = (C_w * delta)**2 * numerator / safe_denominator

    return fno.Math.where(denominator > 0.0, nut, 0.0)

In [7]:
# Define the WALE closure
wale_closure = fno.Closure(
    name="nutFjord",
    operator=fno.Operator.function(wale_function),
    inputs={
        "velocity_grad": fno.Field.grad("U"),
        "delta": fno.Field.delta(),
    },
    outputs={
        "nut": fno.Field("nut"),
    },
)

### Case Definition

In [8]:
# Initialize the OpenFOAM case
case = fno.OpenFOAM.Case(
    name=CASE_NAME,
    case_dir=OF_SCRIPT_DIR,
    run_dir=OUTPUT_DIR,
    of_cmd="module load openfoam/2512",
    shell="bash",
    application="pimpleFoam",
)

case.initialize(ranks=N_TASKS, mesh="blockMesh", validate_mesh=True);

### Submit Job

In [9]:
# Connect the OpenFOAM case and SLURM scheduler (launch a longship instance)
longship = fno.Longship(case=case, closures=(wale_closure,))

# Set sail for the OpenFOAM case (submit the job to the HPC cluster)
run = longship.launch(start_timeout=900)

[FoamNordic] Preparing mesh with blockMesh: pitzDailyWALE
[FoamNordic] Mesh is ready: pitzDailyWALE
[FoamNordic] Sailing in background: pitzDailyWALE


In [10]:
# Wait for the job to complete (polling the job status)
result = run.stop(force=False, timeout=3600, progress=True)

In [11]:
# Summary of the job result
result.summary(style="compact");

Job ID,Name,Status,Partition,Node,Elapsed
-,pitzDailyWALE,succeeded,local,rc5283,00:00:57


### Postprocessing

In [12]:
# Postprocessing
post = result.postprocess

velocity = post.field("U", time_idx=-1)
pressure = post.field("p", time_idx=-1)

print("U shape:", velocity.shape)
print("p shape:", pressure.shape)

statistics = post.statistics(
    ["U", "p", "nut"],
    time_idx=-1,
    verbose=True,
)

U shape: (12225, 3)
p shape: (12225,)


Field,Min,Max,Mean,Std,RMS
U,1.164906e-02,1.346690e+01,6.395988e+00,2.619908e+00,6.911771e+00
p,-3.771660e+01,1.129640e+02,6.460432e+01,2.186080e+01,6.820273e+01
nut,6.524120e-20,1.382990e-04,2.205728e-06,1.084035e-05,1.106248e-05
